# 06 — Scale Comparison

How does the override mechanism evolve from 1B to 27B parameters? (RQ4)

This notebook aggregates Phase 1 results across all four model scales and answers:
- Does the peak differential layer shift in normalized position with scale?
- Do the same feature types (correctness_signal, override_trigger) emerge at all scales?
- Does mean effect size increase with scale — does sycophancy get mechanistically 'deeper' in larger models?
- At what scale do CoT unfaithfulness and sycophancy become geometrically distinguishable?

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.analysis.scale import (
    MODELS,
    MODEL_PARAM_COUNTS,
    MODEL_N_LAYERS,
    normalize_layer_position,
    peak_layer_trend,
)

# Map each model to its Phase 1 run ID
# Fill these in after running Phase 1 on each model scale
SCALE_RUNS = {
    'gemma3_1b': 'FILL_IN_RUN_ID_1B',
    'gemma3_4b': 'FILL_IN_RUN_ID_4B',
    'gemma3_12b': 'FILL_IN_RUN_ID_12B',
    'gemma3_27b': 'FILL_IN_RUN_ID_27B',
}
PHASE1_DIR = Path('../experiments/results/phase1')
DATASET = 'sycophancy_pressure'  # Use the same dataset for cross-scale comparison

In [ ]:
# Load layer summaries for each model scale
scale_summaries = {}
for model_name, run_id in SCALE_RUNS.items():
    if 'FILL_IN' in run_id:
        print(f'Skipping {model_name}: run ID not set')
        continue
    p = PHASE1_DIR / run_id / DATASET / 'layer_summary.json'
    if p.exists():
        with open(p) as f:
            scale_summaries[model_name] = json.load(f)
        print(f'Loaded {model_name}: {len(scale_summaries[model_name])} layers')
    else:
        print(f'Missing: {p}')

In [ ]:
# Plot: normalized peak differential layer position vs. model scale
if scale_summaries:
    peak_fractions = {}
    for model_name, summary in scale_summaries.items():
        n_sig_per_layer = {
            int(l): v.get('n_significant', 0) for l, v in summary.items()
        }
        if not n_sig_per_layer or max(n_sig_per_layer.values()) == 0:
            continue
        peak_layer = max(n_sig_per_layer, key=n_sig_per_layer.get)
        peak_fractions[model_name] = normalize_layer_position(peak_layer, model_name)

    param_counts = [MODEL_PARAM_COUNTS[m] / 1e9 for m in peak_fractions]
    fractions = list(peak_fractions.values())
    labels = [m.replace('gemma3_', '') for m in peak_fractions]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.semilogx(param_counts, fractions, 'o-', color='#2166ac', linewidth=2, markersize=8)
    for x, y, label in zip(param_counts, fractions, labels):
        ax.annotate(label, (x, y), textcoords='offset points', xytext=(5, 5), fontsize=9)
    ax.set_xlabel('Model parameters (billions, log scale)')
    ax.set_ylabel('Normalized peak differential layer position [0, 1]')
    ax.set_title('Does the override mechanism move deeper with scale?')
    ax.set_ylim(0, 1)
    ax.axhline(0.6, color='grey', linestyle='--', linewidth=0.8, label='60% depth')
    ax.legend()
    plt.tight_layout()
    plt.savefig('../paper/figures/scale_peak_layer.png', dpi=150)
    plt.show()

In [ ]:
# Number of significant features vs. scale
if scale_summaries:
    fig, ax = plt.subplots(figsize=(8, 5))
    
    for model_name, summary in scale_summaries.items():
        n_layers = MODEL_N_LAYERS[model_name]
        layers = sorted(int(l) for l in summary.keys())
        normalized_layers = [l / (n_layers - 1) for l in layers]
        n_sig = [summary[str(l)].get('n_significant', 0) for l in layers]
        label = f"{model_name.replace('gemma3_', '')} ({MODEL_PARAM_COUNTS[model_name]/1e9:.0f}B)"
        ax.plot(normalized_layers, n_sig, linewidth=1.5, label=label, alpha=0.8)
    
    ax.set_xlabel('Normalized layer position [0, 1]')
    ax.set_ylabel('# significant differential features (Bonferroni α=0.05)')
    ax.set_title(f'Feature activation profiles across scales — {DATASET}')
    ax.legend()
    plt.tight_layout()
    plt.savefig('../paper/figures/scale_feature_profiles.png', dpi=150)
    plt.show()

In [ ]:
# Mean effect size (Cohen's d) vs. scale
if scale_summaries:
    scale_effect_sizes = {}
    for model_name, summary in scale_summaries.items():
        effects = [v.get('mean_effect_size', 0) for v in summary.values()]
        scale_effect_sizes[model_name] = np.mean([e for e in effects if e > 0])

    params = [MODEL_PARAM_COUNTS[m] / 1e9 for m in scale_effect_sizes]
    effects = list(scale_effect_sizes.values())
    labels = [m.replace('gemma3_', '') for m in scale_effect_sizes]

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.semilogx(params, effects, 'o-', color='#b2182b', linewidth=2, markersize=8)
    for x, y, label in zip(params, effects, labels):
        ax.annotate(label, (x, y), textcoords='offset points', xytext=(5, 3), fontsize=9)
    ax.set_xlabel('Parameters (billions, log scale)')
    ax.set_ylabel("Mean Cohen's d")
    ax.set_title('Does feature differentiation scale with model size?')
    plt.tight_layout()
    plt.savefig('../paper/figures/scale_effect_sizes.png', dpi=150)
    plt.show()

In [ ]:
# Summary table for paper
if scale_summaries:
    rows = []
    for model_name, summary in scale_summaries.items():
        n_layers = MODEL_N_LAYERS[model_name]
        total_sig = sum(v.get('n_significant', 0) for v in summary.values())
        n_sig_per_layer = {int(l): v.get('n_significant', 0) for l, v in summary.items()}
        peak_layer = max(n_sig_per_layer, key=n_sig_per_layer.get) if n_sig_per_layer else 0
        rows.append({
            'Model': model_name.replace('gemma3_', 'Gemma 3 '),
            'Params': f"{MODEL_PARAM_COUNTS[model_name]/1e9:.0f}B",
            'Layers': n_layers,
            'Total sig. features': total_sig,
            'Peak layer': peak_layer,
            'Peak (normalized)': f"{normalize_layer_position(peak_layer, model_name):.2f}",
        })
    print(pd.DataFrame(rows).to_string(index=False))